# SQL query from table names - Continued

In [1]:
from google.colab import userdata
from openai import OpenAI


OPENAI_API_KEY = userdata.get("OPENAI_API_KEY").strip()


client = OpenAI(api_key=OPENAI_API_KEY)

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [12]:
context = [ {'role':'system', 'content':"""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    country TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    order_date TEXT
);
"""} ]



In [13]:
context.append( {'role':'system', 'content':"""
-- Maintain the SQL order simple and efficient as you can, using valid SQLite, answer the following questions for the tables provided above.

Example 1:
Question: Show all customers from France.
SQL:
SELECT * FROM customers
WHERE country = 'France';

Example 2:
Question: Show all orders with the customer name.
SQL:
SELECT o.order_id, c.name, o.product_id, o.quantity, o.order_date
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id;

Example 3:
Question: Show product names and prices for products in the Electronics category.
SQL:
SELECT product_name, price
FROM products
WHERE category = 'Electronics';
"""
})

In [14]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [17]:
#new
context_user = context.copy()
print(return_CCRMSQL(
    "Show all customers from Germany who ordered products in the Electronics category.",
    context_user
))

SQL:
SELECT c.name, c.city, c.country
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN products p ON o.product_id = p.product_id
WHERE c.country = 'Germany' AND p.category = 'Electronics';


In [18]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Show all customers from Germany who ordered products in the Electronics category", old_context_user))

This is your SQL:
```sql
SELECT customers.name
FROM customers
JOIN orders ON customers.ID = orders.customer_ID
JOIN products ON orders.product_ID = products.ID
WHERE customers.country = 'Germany' AND products.category = 'Electronics';
```

This SQL query retrieves the names of customers from Germany who have placed orders for products in the Electronics category. It uses JOIN to connect the customers, orders, and products tables based on their respective IDs, and then filters the results to only include customers from Germany and products in the Electronics category.


In [22]:
#new
print(return_CCRMSQL(
    "Calculate the total revenue for each product.",
    context_user
))

SQL:
SELECT p.product_id, p.product_name, SUM(o.quantity * p.price) AS total_revenue
FROM products p
JOIN orders o ON p.product_id = o.product_id
GROUP BY p.product_id, p.product_name;


In [23]:
#old
print(return_CCRMSQL(
    "Calculate the total revenue for each product.",
    old_context_user
))

This is your SQL:
```sql
SELECT product_id, SUM(price) AS total_revenue
FROM sales
GROUP BY product_id;
```

This SQL query selects the product ID and calculates the total revenue by summing the price for each product from the sales table. The result is grouped by product ID to show the total revenue for each product.


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [24]:
#new
print(return_CCRMSQL(
    "Show the total number of orders for each customer.",
    context_user
))

SQL:
SELECT c.name, COUNT(o.order_id) AS total_orders
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id;


In [25]:
#old
print(return_CCRMSQL(
    "Show the total number of orders for each customer.",
    old_context_user
))

This is your SQL:
```sql
SELECT ID_usr, COUNT(*) AS total_orders
FROM orders
GROUP BY ID_usr;
```

This SQL query selects the customer ID and counts the total number of orders for each customer by grouping the results based on the customer ID.


In this lab I compared the old prompt with the new prompt that includes context and examples.

First, I created the database structure with three tables: customers, orders, and products. Then I added few-shot examples to guide the model.

I tested different questions like filtering customers, calculating revenue, and counting orders per customer.

The new prompt worked better in most cases. It produced more complete SQL queries, especially when joins and group by were needed. The structure of the queries was also more correct.

The old prompt sometimes missed joins or produced simpler and less accurate queries. It looks like the model had to guess more without examples.

I learned that adding examples in the prompt helps the model understand how to write SQL queries. Clear structure and guidance improve the results a lot.